# Document Intelligence Refinery — Run in Colab

This notebook clones  repo (or uses the current folder), installs deps, and runs the interim pipeline: **DocumentProfiles** + **extraction_ledger.jsonl**.

**Before running:** Upload your `data.zip` (PDFs inside a `data/` folder) when prompted, or use a repo that already has `data/`.

## 1. Clone repo from GitHub (or skip if you opened from GitHub)

Paste your repo URL below and run. If you opened this notebook from GitHub, you can skip to section 2.

In [3]:
# Paste your GitHub repo URL
REPO_URL = "https://github.com/zumi123/document_intelligence_refinery.git"

import os
if not os.path.exists("/content/document_intelligence_refinery/src"):
    !git clone {REPO_URL} /content/document_intelligence_refinery
    %cd /content/document_intelligence_refinery
else:
    %cd /content/document_intelligence_refinery
    !pwd

Cloning into '/content/document_intelligence_refinery'...
remote: Enumerating objects: 137, done.
remote: Counting objects: 100% (137/137), done.
remote: Compressing objects: 100% (92/92), done.
remote: Total 137 (delta 49), reused 120 (delta 33), pack-reused 0 (from 0)
Receiving objects: 100% (137/137), 77.54 KiB | 7.05 MiB/s, done.
Resolving deltas: 100% (49/49), done.
/content/document_intelligence_refinery


## 2. Install dependencies

In [2]:
%cd /content/document_intelligence_refinery
!pip install -e . -q
!pip install pymupdf pyyaml -q
# Docling is heavy; install only if you need Strategy B / Phase 0 Docling
!pip install docling -q
print("Done.")

/content/document_intelligence_refinery
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for document-intelligence-refinery (pyproject.toml) ... done
Done.


## 3. Optional: Use Colab TPU (or GPU) for faster embeddings


In [3]:
# Use GPU or TPU.
import os
USE_TPU = "COLAB_TPU_ADDR" in os.environ

if USE_TPU:
    !pip install -q torch_xla
    try:
        import torch_xla.core.xla_model as xm
        _tpu_dev = xm.xla_device()
        print("TPU detected. Note: sentence-transformers has limited TPU support; embedding may fall back to CPU.")
        DEVICE = _tpu_dev
    except Exception as e:
        print("TPU setup failed:", e, "- using CPU for embeddings.")
        import torch
        DEVICE = torch.device("cpu")
else:
    import torch
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Device for embeddings:", DEVICE)

!pip install -q sentence-transformers

class _ColabEmbeddingFn:
    """ChromaDB embedding function using GPU (or CPU fallback). For TPU, support is limited."""

    def __init__(self, device):
        self._device = device
        self._model = None

    def name(self):
        return "colab_sentence_transformers"
    def _model_init(self):
        if self._model is None:
            from sentence_transformers import SentenceTransformer
            self._model = SentenceTransformer("all-MiniLM-L6-v2")
            # sentence_transformers works best on cuda/cpu; for TPU use cpu fallback
            if str(self._device).startswith("xla"):
                self._model = self._model.to("cpu")
            else:
                self._model = self._model.to(self._device)
    def __call__(self, input):
        self._model_init()
        dev = self._device if not str(self._device).startswith("xla") else "cpu"
        texts = input if isinstance(input, list) else [input]
        embs = self._model.encode(texts, convert_to_numpy=True, device=dev)
        return embs.tolist()

    def embed_query(self, input):
        """ChromaDB uses this to embed the search query. Returns list of one vector (query_embeddings format)."""
        inp = [input] if not isinstance(input, list) else input
        return self(inp)

EMBEDDING_FN = _ColabEmbeddingFn(DEVICE)
print("Embedding function ready. Run the 'Final pipeline with GPU/TPU embeddings' cell below to use it.")

Device for embeddings: cuda
Embedding function ready. Run the 'Final pipeline with GPU/TPU embeddings' cell below to use it.


## 4. Phase 0: Character density analysis (pdfplumber)

In [4]:
%cd /content/document_intelligence_refinery
!python scripts/phase0_pdfplumber_analysis.py

/content/document_intelligence_refinery
Phase 0: Character density / bbox / image-area analysis (pdfplumber or PyMuPDF)

--- Consumer Price Index July 2025.pdf ---
  Pages: 13, Chars/page (avg): 1041.6, Char density: 0.002149, Image ratio: 0.1593
  Min/max chars per page: 376 / 2133

--- tax_expenditure_ethiopia_2021_22.pdf ---
  Pages: 60, Chars/page (avg): 1478.9, Char density: 0.002952, Image ratio: 0.0007
  Min/max chars per page: 104 / 2743

--- 2013-E.C-Audit-finding-information.pdf ---
  Pages: 3, Chars/page (avg): 0.0, Char density: 0.0, Image ratio: 1.0
  Min/max chars per page: 0 / 0

Results written to: /content/document_intelligence_refinery/scripts/phase0_pdfplumber_results.txt


In [5]:
%cd /content/document_intelligence_refinery
!python scripts/phase0_docling_analysis.py

/content/document_intelligence_refinery
2026-03-09 09:05:49.746152: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773047149.767340   31584 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773047149.774688   31584 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773047149.792879   31584 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773047149.792905   31584 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773047149.792909   31584 computati

## 5. Generate profiles + extraction ledger (interim deliverables)

In [6]:
%cd /content/document_intelligence_refinery
!python scripts/run_interim_artifacts.py

/content/document_intelligence_refinery
Triage + extract: Consumer Price Index July 2025.pdf
  -> fast_text, confidence=0.90
Triage + extract: tax_expenditure_ethiopia_2021_22.pdf
2026-03-09 09:09:01.333670: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773047341.354876   32363 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773047341.361982   32363 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773047341.379942   32363 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773047341.379970   32363 computation_placer.cc:177] computation place

## 6. Run tests

In [8]:
%cd /content/document_intelligence_refinery
!pip install -e ".[dev]" -q
!pytest tests/ -v --tb=short

/content/document_intelligence_refinery
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for document-intelligence-refinery (pyproject.toml) ... done
============================= test session starts ==============================
platform linux -- Python 3.12.12, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/document_intelligence_refinery
configfile: pyproject.toml
plugins: Faker-40.8.0, cov-7.0.0, typeguard-4.5.1, anyio-4.12.1, langsmith-0.7.7
collected 16 items                                                             

tests/test_chunker.py::test_rule1_valid_table_passes PASSED              [  6%]
tests/test_chunker.py::test_rule1_breaks_when_table_split_from_header PASSED [ 12%]
tests/test_chunker.py::test_rule4_breaks_when_parent_section_missing PASSED [ 18%]


## 7. Download artifacts

Zip `.refinery/` and download it to your machine.

In [9]:
%cd /content/document_intelligence_refinery
!zip -r refinery_artifacts.zip .refinery
from google.colab import files
files.download("refinery_artifacts.zip")

/content/document_intelligence_refinery
updating: .refinery/ (stored 0%)
updating: .refinery/extraction_ledger.jsonl (deflated 61%)
updating: .refinery/profiles/ (stored 0%)
updating: .refinery/profiles/363d57b988a034fa.json (deflated 38%)
updating: .refinery/profiles/2c780c75aade84e8.json (deflated 37%)
updating: .refinery/chromadb/ (stored 0%)
updating: .refinery/facts.db (deflated 98%)
updating: .refinery/qa_examples/ (stored 0%)
updating: .refinery/profiles/fd9b7db399992e41.json (deflated 38%)
updating: .refinery/pageindex/ (stored 0%)
updating: .refinery/pageindex/2c780c75aade84e8.json (deflated 40%)
updating: .refinery/pageindex/fd9b7db399992e41.json (deflated 40%)
updating: .refinery/.ipynb_checkpoints/ (stored 0%)
updating: .refinery/profiles/.ipynb_checkpoints/ (stored 0%)
updating: .refinery/pageindex/.ipynb_checkpoints/ (stored 0%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Run final pipeline with GPU/TPU embeddings

In [10]:
# Run final pipeline with custom embedding (GPU/TPU). Requires section 5 done.
%cd /content/document_intelligence_refinery
import json
from pathlib import Path
project_root = Path("/content/document_intelligence_refinery")
from src.agents.triage import triage_document
from src.agents.extractor import ExtractionRouter
from src.agents.chunker import ChunkingEngine
from src.agents.indexer import build_page_index
from src.data.vector_store import VectorStore
from src.data.fact_table import FactTableExtractor
from src.agents.query_agent import QueryAgent
from src.data.audit import verify_claim
INTERIM_DOCS = [ "Consumer Price Index July 2025.pdf", "tax_expenditure_ethiopia_2021_22.pdf"]
data_dir = project_root / "data"
pageindex_dir = project_root / ".refinery" / "pageindex"
qa_dir = project_root / ".refinery" / "qa_examples"
ledger_path = project_root / ".refinery" / "extraction_ledger.jsonl"
pageindex_dir.mkdir(parents=True, exist_ok=True)
qa_dir.mkdir(parents=True, exist_ok=True)
router = ExtractionRouter(ledger_path=ledger_path)
chunker = ChunkingEngine()
# Use GPU/TPU embeddings only if you ran section 3 (EMBEDDING_FN); else default CPU
embedding_fn = globals().get("EMBEDDING_FN")
vector_store = VectorStore(embedding_function=embedding_fn)
fact_extractor = FactTableExtractor(project_root / ".refinery" / "facts.db")
query_agent = QueryAgent(pageindex_dir=pageindex_dir, vector_store=vector_store, fact_table=fact_extractor)
for name in INTERIM_DOCS:
    path = data_dir / name
    if not path.exists(): continue
    print("Processing:", name)
    try:
        profile = triage_document(path)
        result = router.extract(path, profile)
        ldus = chunker.chunk(result.extracted)
        pi = build_page_index(profile.document_id, ldus, result.extracted)
        with open(pageindex_dir / f"{profile.document_id}.json", "w") as f: json.dump(pi.model_dump(mode="json"), f, indent=2)
        vector_store.ingest(ldus, profile.document_id)
        fact_extractor.ingest(profile.document_id, ldus)
    except Exception as e: print("  Error:", e)
print("Done." + (" ChromaDB used GPU/TPU embeddings." if embedding_fn else " ChromaDB used default (CPU) embeddings."))

/content/document_intelligence_refinery
Processing: Consumer Price Index July 2025.pdf


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Processing: tax_expenditure_ethiopia_2021_22.pdf


[INFO] 2026-03-09 09:13:23,121 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-03-09 09:13:23,137 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-03-09 09:13:23,138 [RapidOCR] main.py:53: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-03-09 09:13:23,252 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-03-09 09:13:23,257 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-03-09 09:13:23,258 [RapidOCR] main.py:53: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-03-09 09:13:23,310 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-03-09 09:13:23,345 [RapidOCR] download_file.py:60: File exists and is valid: /usr/loc

Done. ChromaDB used GPU/TPU embeddings.


## Demo cells: Triage, Query, Audit

Run these **after** the final pipeline (section 8) has completed. They show: (1) Triage on one PDF, (2) Query with provenance, (3) Audit mode. Paths are set for Colab (`/content/document_intelligence_refinery`).

In [12]:
from pathlib import Path
ROOT = Path("/content/document_intelligence_refinery")

# DEMO 1: Triage
from src.agents.triage import triage_document
p = ROOT / "data" / "Consumer Price Index July 2025.pdf"
profile = triage_document(p)
print("Triage:", profile.model_dump_json(indent=2))

Triage: {
  "document_id": "2c780c75aade84e8",
  "origin_type": "native_digital",
  "layout_complexity": "single_column",
  "language_code": "en",
  "language_confidence": 0.9,
  "domain_hint": "general",
  "estimated_extraction_cost": "fast_text_sufficient",
  "num_pages": 13,
  "chars_per_page_avg": 1041.6,
  "image_area_ratio_avg": 0.1593
}


In [13]:
# DEMO 2: Query with provenance
from src.agents.query_agent import QueryAgent
from src.data.vector_store import VectorStore
from src.data.fact_table import FactTableExtractor
# ChromaDB needs an embedding function to embed the query; use same as pipeline or default
_ef = globals().get("EMBEDDING_FN")
if _ef is None:
    import chromadb.utils.embedding_functions as _efmod
    _ef = _efmod.DefaultEmbeddingFunction()
vs = VectorStore(ROOT / ".refinery" / "chromadb", embedding_function=_ef)
ft = FactTableExtractor(ROOT / ".refinery" / "facts.db")
agent = QueryAgent(ROOT / ".refinery" / "pageindex", vs, ft)
answer, prov = agent.answer("What was July inflation rate?", profile.document_id)
print("Answer:", answer[:500] if answer else "(none)")
print("Provenance:", prov.model_dump_json(indent=2))


Answer: According to **page 1** (Document):
The
major contributing factors for the rise of food inflation for July EFY2017 was the observed
increase in the average prices of major food commodities such as bread and cereals (-1.9%),
vegetables (18.7%), meat (12.1%), Sugar, jam, honey & chocolat

According to **page 1** (Document):
ETHIOPIAN STATISTICAL SERVICE
Statistical Bulletin
MONTHLY NEWS RELEASE
INFLATION REPORT ON JULY, EFY 2017
Issue No. 25/2017 July EFY2017 Page 1
1. Summary
The Consumer Price I
Provenance: {
  "citations": [
    {
      "document_name": "2c780c75aade84e8",
      "page_number": 1,
      "bbox": null,
      "content_hash": "18d8a18bdd4b0cee",
      "content_snippet": "The\nmajor contributing factors for the rise of food inflation for July EFY2017 was the observed\nincrease in the average prices of major food commodities such as bread and cereals (-1.9%),\nvegetables ("
    },
    {
      "document_name": "2c780c75aade84e8",
      "page_number": 1,
      "bbox": 

In [14]:
# DEMO 3: Audit mode
from src.data.audit import audit_mode
result = audit_mode("The report states the monthly consumer price data collection first started in Addis Ababa", vs, ft, profile.document_id)
print("Audit:", result.model_dump_json(indent=2))

Audit: {
  "claim": "The report states the monthly consumer price data collection first started in Addis Ababa",
  "decision": "verified",
  "citations": [
    {
      "document_name": "2c780c75aade84e8",
      "page_number": 9,
      "bbox": null,
      "content_hash": "8b730f8e205bba83",
      "content_snippet": "ETHIOPIAN STATISTICAL SERVICE\nStatistical Bulletin\nIssue No. 25/2017 Page 9\nRecent Improvements in CPI Price survey\nThe monthly consumer price data collection first started in Addis Ababa in 1963 (195"
    },
    {
      "document_name": "2c780c75aade84e8",
      "page_number": 9,
      "bbox": null,
      "content_hash": "58d7a3db2130f9a2",
      "content_snippet": "This exercise was a milestone for the next CPI re-basing which is planned for December 2025=100 using the weights to be derived\nfrom 2017EFY National Integrated Household Survey. Furthermore, these 80"
    }
  ],
  "rationale": "Found 2 supporting citation(s) in LDUs and fact table.",
  "evidence_summary": 

In [18]:
!pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 73.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 105.7 MB/s eta 0:00:00


In [19]:
%cd /content/document_intelligence_refinery
!pip install -e ".[ui]" -q

/content/document_intelligence_refinery
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for document-intelligence-refinery (pyproject.toml) ... done


In [2]:
import subprocess
import threading
import time

def run_streamlit():
    subprocess.run(
        ["streamlit", "run", "app_ui.py", "--server.port=8501", "--server.headless=true", "--browser.gatherUsageStats=false"],
        cwd="/content/document_intelligence_refinery",
    )

thread = threading.Thread(target=run_streamlit, daemon=True)
thread.start()
time.sleep(6)
print("Streamlit is starting. Run the next cell to get the public URL (Cloudflare Tunnel, no password).\n")

Exception in thread Thread-3 (run_streamlit):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/ipykernel_161/508492083.py", line 6, in run_streamlit
  File "/usr/lib/python3.12/subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "/usr/lib/python3.12/subprocess.py", line 1955, in _execute_child
    raise child_exception_type(errno_num, err_msg, err_filename)
FileNotFoundError: [Errno 2] No such file or directory: '/content/document_intelligence_refinery'


Streamlit is starting. Run the next cell to get the public URL (Cloudflare Tunnel, no password).



In [1]:
# Expose port 8501 with Cloudflare Tunnel (no signup, no password). Keep this cell running.
!curl -sL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /tmp/cloudflared && chmod +x /tmp/cloudflared
!/tmp/cloudflared tunnel --url http://localhost:8501

2026-03-09T09:54:22Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-03-09T09:54:22Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-03-09T09:54:25Z INF +--------------------------------------------------------------------------------------------+
2026-03-09T09:54:25Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-03-09T09:54:25Z INF |  https://result-robot-affiliate-vegetarian.trycloudfla